# 개념 3. 칼만 필터

**목표:** 흔들리는 측정과 *예측*을 섞어 더 믿을 만한 값을 만드는 칼만 필터를 이해한다. 그리고 그 안의 *예측* 단계가 바로 **선형 외삽**이라는 것을 본다.

순서
1. 가장 단순한 칼만 필터: 값 하나 추정하기
2. 움직임 넣기: 예측(= 선형 외삽) + 보정
3. 재지 않은 속도까지 나온다
4. 한계: 직선 관계만 다룬다

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"   # 한글 글꼴 (macOS)
plt.rcParams["axes.unicode_minus"] = False

## 1. 가장 단순한 칼만 필터

등대까지 거리를 흔들리는 거리계로 여러 번 잰다. 진짜 거리는 100m, 측정은 ±10m 흔들린다.

칼만 필터는 두 가지를 들고 다닌다.

| 기호 | 뜻 |
|---|---|
| $x$ | 지금 추정값 |
| $P$ | 그 추정이 얼마나 틀렸을 수 있나 (분산) |

측정 $z$ (흔들림 분산 $R$)가 오면

$$K = \frac{P}{P + R}, \qquad x \leftarrow x + K(z - x), \qquad P \leftarrow (1 - K)P$$

- $K$(칼만 이득)는 *측정을 얼마나 믿을지*다. 0이면 무시, 1이면 측정 그대로.
- 추정이 불확실하면($P$ 큼) 측정을 많이 믿고, 확실해지면 조금만 반영한다.

In [ ]:
rng = np.random.default_rng(0)
z = 100 + rng.normal(0, 10, 30)     # 측정 30번
R = 10**2

x, P = z[0], R                      # 첫 측정으로 시작
xs, Ks = [x], [1.0]
for zk in z[1:]:
    K = P / (P + R)
    x = x + K * (zk - x)
    P = (1 - K) * P
    xs.append(x); Ks.append(K)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(z, "o", color="gray", label="측정")
axes[0].plot(xs, "-", lw=2, label="칼만 추정")
axes[0].axhline(100, color="k", ls=":", label="진짜 값")
axes[0].set_xlabel("측정 횟수"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(Ks, "-o", ms=3)
axes[1].set_xlabel("측정 횟수"); axes[1].set_ylabel("K"); axes[1].set_title("칼만 이득: 점점 측정을 덜 믿는다")
axes[1].grid(alpha=.3); plt.show()

값이 변하지 않으면 이건 *평균*과 같다. 칼만 필터의 힘은 값이 **움직일 때** 나온다.

## 2. 움직임 넣기: 예측 + 보정

직선 위를 움직이는 물체를 1초마다 위치만 잰다(±3m). 상태를 **위치와 속도** 두 개로 둔다: $\mathbf{x} = [x,\ v]$.

**예측** ($dt$초 앞으로): *"같은 속도로 간다"*고 보고 옮긴다. 개념 1의 **선형 외삽** 그대로다.

$$\mathbf{x}^- = F\mathbf{x},\quad F = \begin{bmatrix}1 & dt\\0 & 1\end{bmatrix} \quad\Longleftrightarrow\quad x^- = x + v\,dt,\ \ v^- = v$$

$$P^- = FPF^\top + Q$$

$Q$는 *"같은 속도라는 가정이 완전히 맞지는 않다"*는 몫이다. 예측하는 동안 불확실성이 커진다.

**보정** (위치 $z$만 잼, $H = [1\ \ 0]$):

$$K = P^-H^\top(HP^-H^\top + R)^{-1},\qquad \mathbf{x} = \mathbf{x}^- + K(z - H\mathbf{x}^-),\qquad P = (I - KH)P^-$$

정리하면 칼만 필터는 **"선형 외삽으로 예측하고, 측정이 오면 믿을 만한 만큼만 고친다"**를 반복한다.

In [ ]:
dt, R = 1.0, 3.0**2
t = np.arange(40)
true_x = 2.0 * t                       # 실제: 2 m/s 등속
z = true_x + rng.normal(0, 3, len(t))  # 위치만 잼

F = np.array([[1, dt], [0, 1]])
Q = 0.01 * np.array([[dt**4 / 4, dt**3 / 2], [dt**3 / 2, dt**2]])
H = np.array([[1.0, 0.0]])

x = np.array([z[0], 0.0]); P = np.diag([R, 10.0])
est = []
for zk in z:
    x = F @ x; P = F @ P @ F.T + Q                       # 예측 (선형 외삽)
    S = H @ P @ H.T + R; K = P @ H.T / S                 # 보정
    x = x + (K * (zk - H @ x)).ravel(); P = (np.eye(2) - K @ H) @ P
    est.append(x.copy())
est = np.array(est)

## 3. 재지 않은 속도까지 나온다

속도는 한 번도 재지 않았다. 그런데 필터는 속도를 추정한다. $P$ 안에 *"위치가 예측보다 앞서 있으면 속도도 더 빠를 것"*이라는 관계(공분산)가 쌓이기 때문이다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t, z, "o", color="gray", ms=4, label="측정 위치")
axes[0].plot(t, est[:, 0], lw=2, label="추정 위치"); axes[0].plot(t, true_x, "k:", label="실제")
axes[0].legend(); axes[0].grid(alpha=.3); axes[0].set_xlabel("초")
axes[1].plot(t, est[:, 1], lw=2, label="추정 속도 (재지 않음!)"); axes[1].axhline(2, color="k", ls=":", label="실제 2 m/s")
axes[1].legend(); axes[1].grid(alpha=.3); axes[1].set_xlabel("초"); plt.show()

다듬은 위치·속도로 앞을 외삽하면, 흔들리는 마지막 두 측정으로 외삽하는 것보다 덜 흔들린다(개념 2의 "여러 점으로"와 같은 효과).

## 4. 한계: 직선 관계만 다룬다

칼만 필터의 예측은 $\mathbf{x}^- = F\mathbf{x}$, 곧 **곱하고 더하기**뿐이다. 그래서 불확실성도 $FPF^\top$로 정확히 옮길 수 있다.

하지만 *"일정하게 돈다"* 같은 움직임에는 $\sin$, $\cos$이 들어간다. 어떤 행렬 $F$를 곱해도 이 식은 나오지 않는다(**비선형**). → **확장 칼만 필터(개념 4)**

## 직접 해 보기

1. 2절의 `Q` 앞 `0.01`을 `0.0001`, `1.0`으로 바꿔 보자. 작으면 예측과 측정 중 무엇을 더 믿나?
2. `R`을 `1.0**2`로 줄이면 추정 위치는 측정에 더 붙나, 덜 붙나?

In [ ]:
# 여기서 직접 바꿔 보세요


## 정리

- 칼만 필터 = **예측**(선형 외삽) + **보정**(측정을 믿을 만한 만큼 반영)의 반복
- 측정하지 않은 값(속도)도 추정한다.
- 직선 관계(행렬 곱)인 움직임만 다룬다.